In [3]:
from google.colab import drive

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.9 MB/s eta 0:00:00


In [6]:
import torch

In [7]:
print("GPU dispo :", torch.cuda.is_available())

GPU dispo : True


In [8]:
import os

In [9]:
print(os.listdir('/content/drive/MyDrive/ml-internship/GTSDB_Train_and_Test'))

['ReadME__(yes actually take a look).txt', 'Train', 'Test']


In [10]:
import yaml
from ultralytics import YOLO
import time
import matplotlib.pyplot as plt

BASE = "/content/drive/MyDrive/ml-internship/GTSDB_Train_and_Test"

class_names = [
    "Limite 20", "Limite 30", "Limite 50", "Limite 60", "Limite 70", "Limite 80",
    "Fin limite 80", "Limite 100", "Limite 120", "Interdit dépassement",
    "Interdit dépassement (poids lourds)", "Priorité intersection", "Route prioritaire",
    "Cédez le passage", "Stop", "Circulation interdite", "Interdit poids lourds",
    "Sens interdit", "Danger général", "Virage dangereux (gauche)", "Virage dangereux (droite)",
    "Double virage", "Route cahoteuse", "Route glissante", "Chaussée rétrécie (droite)",
    "Travaux", "Feux tricolores", "Passage piétons", "Enfants", "Cyclistes",
    "Neige/verglas", "Passage animaux", "Fin de restrictions", "Direction obligatoire (droite)",
    "Direction obligatoire (gauche)", "Direction obligatoire (tout droit)",
    "Tout droit ou droite", "Tout droit ou gauche", "Contournement (droite)",
    "Contournement (gauche)", "Rond-point obligatoire", "Fin interdit dépassement",
    "Fin interdit dépassement (poids lourds)",
]

data_config = {
    "train": f"{BASE}/Train/images",
    "val": f"{BASE}/Test/images",
    "nc": len(class_names),
    "names": class_names,
}

with open("/content/gtsdb.yaml", "w") as f:
    yaml.dump(data_config, f, allow_unicode=True)

print(f"{len(class_names)} classes configurées.")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
43 classes configurées.


In [11]:
EPOCHS = 50
IMG_SIZE = 640

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/gtsdb.yaml",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=16,
    project="/content/drive/MyDrive/ml-internship/yolo_runs",
    name="gtsdb_v1",
    verbose=False,
)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gtsdb.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=gtsdb_v1, nbs=64, nms=False, opset=None, o

In [12]:
import os, shutil

BASE = "/content/drive/MyDrive/ml-internship/GTSDB_Train_and_Test"

# ancien id -> nouveau groupe (0 à 5)
GROUP_MAP = {}
for i in range(0, 9):   GROUP_MAP[i] = 0   # limites de vitesse
for i in [9, 10, 32, 41, 42]: GROUP_MAP[i] = 1  # restriction dépassement
for i in [11, 12, 13, 14]: GROUP_MAP[i] = 2  # priorité
for i in [15, 16, 17]: GROUP_MAP[i] = 3  # circulation interdite
for i in range(18, 32): GROUP_MAP[i] = 4  # danger / avertissement
for i in range(33, 41): GROUP_MAP[i] = 5  # direction obligatoire

group_names = [
    "Limite de vitesse",
    "Restriction depassement",
    "Priorite",
    "Circulation interdite",
    "Danger / Avertissement",
    "Direction obligatoire",
]

def regroup_split(split):
    labels_dir = f"{BASE}/{split}/labels"
    backup_dir = f"{BASE}/{split}/labels_43_original"

    if not os.path.exists(backup_dir):
        shutil.copytree(labels_dir, backup_dir)
        print(f"Backup créé : {backup_dir}")

    for fname in os.listdir(backup_dir):
        if not fname.endswith(".txt"):
            continue
        new_lines = []
        with open(f"{backup_dir}/{fname}") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                new_id = GROUP_MAP.get(int(parts[0]))
                if new_id is None:
                    continue  # ignore les ids hors mapping (ex: le label 100 corrompu vu en local)
                new_lines.append(" ".join([str(new_id)] + parts[1:]))
        with open(f"{labels_dir}/{fname}", "w") as f:
            f.write("\n".join(new_lines))

regroup_split("Train")
regroup_split("Test")

# supprime les caches Ultralytics périmés (ils contiennent encore les anciens ids)
import glob
for cache_file in glob.glob(f"{BASE}/*/*.cache"):
    os.remove(cache_file)
    print("Cache supprimé :", cache_file)

print("Régroupement terminé — 43 → 6 classes.")

Backup créé : /content/drive/MyDrive/ml-internship/GTSDB_Train_and_Test/Train/labels_43_original
Backup créé : /content/drive/MyDrive/ml-internship/GTSDB_Train_and_Test/Test/labels_43_original
Cache supprimé : /content/drive/MyDrive/ml-internship/GTSDB_Train_and_Test/Train/labels.cache
Cache supprimé : /content/drive/MyDrive/ml-internship/GTSDB_Train_and_Test/Test/labels.cache
Régroupement terminé — 43 → 6 classes.


In [13]:
data_config = {
    "train": f"{BASE}/Train/images",
    "val": f"{BASE}/Test/images",
    "nc": 6,
    "names": group_names,
}
with open("/content/gtsdb_grouped.yaml", "w") as f:
    yaml.dump(data_config, f, allow_unicode=True)

model_v2 = YOLO("yolov8n.pt")  # nouveau modèle : le nombre de classes a changé, impossible de reprendre le checkpoint précédent

results_v2 = model_v2.train(
    data="/content/gtsdb_grouped.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/drive/MyDrive/ml-internship/yolo_runs",
    name="gtsdb_v2_grouped",
    verbose=False,
)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gtsdb_grouped.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=gtsdb_v2_grouped, nbs=64, nms=Fals

In [14]:
model_v3 = YOLO("yolov8s.pt")  # version "small" — plus de capacité que "nano"

results_v3 = model_v3.train(
    data="/content/gtsdb_grouped.yaml",  # même config 6 classes que l'essai précédent
    epochs=100,
    imgsz=640,
    batch=16,
    patience=30,  # arrête tout seul si ça stagne, pas besoin d'attendre 100 pour rien
    project="/content/drive/MyDrive/ml-internship/yolo_runs",
    name="gtsdb_v3_yolov8s",
    verbose=False,
)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gtsdb_grouped.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=gtsdb_v3_yolov8s, nbs=64, nms=Fal